In [1]:
import pandas as pd

# World Bank exports have 4 metadata rows before the real header
df = pd.read_csv("world_bank_raw.csv", skiprows=4)

df.head()  # sanity check

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,37524.914920,39287.059713,38554.716783,28733.604511,35207.170909,41863.273996,47521.876547,51712.153136,NaN,NaN
1,Africa Eastern and Southern,AFE,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,3843.028379,3727.710785,3804.533834,3707.033612,4022.277659,4359.591407,4496.744898,4635.125049,4840.265200,NaN
2,Afghanistan,AFG,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,2335.795862,2432.276701,2583.485332,2561.981761,2144.166570,2122.659362,2203.537619,2236.199073,NaN,NaN
3,Africa Western and Central,AFW,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,5035.762391,5188.368000,5491.687935,5423.594405,5702.134886,6232.323014,6538.812038,6840.333882,7184.664483,NaN
4,Angola,AGO,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,8125.706394,8370.228758,8621.659308,7782.151534,8720.187687,9373.408309,9549.793139,9963.590209,10251.345706,NaN


In [2]:
# Build the list of year columns we actually need: 2017 through 2025
year_cols = [str(y) for y in range(2017, 2026)]

# Keep the identifier columns plus only those years
keep_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"] + year_cols
df = df[keep_cols]

df.head()  # confirm the trim worked

,Country Name,Country Code,Indicator Name,Indicator Code,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,37524.914920,39287.059713,38554.716783,28733.604511,35207.170909,41863.273996,47521.876547,51712.153136,NaN
1,Africa Eastern and Southern,AFE,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,3843.028379,3727.710785,3804.533834,3707.033612,4022.277659,4359.591407,4496.744898,4635.125049,4840.265200
2,Afghanistan,AFG,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,2335.795862,2432.276701,2583.485332,2561.981761,2144.166570,2122.659362,2203.537619,2236.199073,NaN
3,Africa Western and Central,AFW,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,5035.762391,5188.368000,5491.687935,5423.594405,5702.134886,6232.323014,6538.812038,6840.333882,7184.664483
4,Angola,AGO,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,8125.706394,8370.228758,8621.659308,7782.151534,8720.187687,9373.408309,9549.793139,9963.590209,10251.345706


In [3]:
# See how many missing values exist per year column
df[year_cols].isna().sum()

# Result: ~19/year through 2022, rising to 33 in 2025.
# Investigated and confirmed: aligns with conflict/instability in specific
# countries (Yemen, South Sudan, Venezuela) and normal reporting lag for
# the most recent years. Decision: leave as true nulls — no imputing,
# no dropping rows. See docs/errors_and_fixes.md.

,0
2017,19
2018,19
2019,19
2020,19
2021,19
2022,19
2023,21
2024,23
2025,33


In [4]:
# Turn 9 year-columns into a single "year" column, one row per country-year.
# Matches OECD's structure and makes SQL joins on year/country possible.
df_long = df.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    value_vars=year_cols,
    var_name="year",
    value_name="gdp_per_capita_ppp"
)

df_long.head(10)  # confirm: same country repeating, one row per year

,Country Name,Country Code,Indicator Name,Indicator Code,year,gdp_per_capita_ppp
0,Aruba,ABW,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,2017,37524.914920
1,Africa Eastern and Southern,AFE,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,2017,3843.028379
2,Afghanistan,AFG,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,2017,2335.795862
3,Africa Western and Central,AFW,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,2017,5035.762391
4,Angola,AGO,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,2017,8125.706394
5,Albania,ALB,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,2017,14110.672057
6,Andorra,AND,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,2017,53084.863964
7,Arab World,ARB,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,2017,14494.514460
8,United Arab Emirates,ARE,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,2017,70281.896942
9,Argentina,ARG,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,2017,23385.074090


In [6]:
df_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2385 entries, 0 to 2384
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country Name        2385 non-null   object 
 1   Country Code        2385 non-null   object 
 2   Indicator Name      2385 non-null   object 
 3   Indicator Code      2385 non-null   object 
 4   year                2385 non-null   object 
 5   gdp_per_capita_ppp  2194 non-null   float64
dtypes: float64(1), object(5)
memory usage: 111.9+ KB


In [7]:
# Convert year from text to an actual whole number, so it can be
# filtered/sorted numerically (e.g. year >= 2020) rather than as text
df_long["year"] = df_long["year"].astype(int)

df_long.info()  # confirm: year should now show as int64, not object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2385 entries, 0 to 2384
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country Name        2385 non-null   object 
 1   Country Code        2385 non-null   object 
 2   Indicator Name      2385 non-null   object 
 3   Indicator Code      2385 non-null   object 
 4   year                2385 non-null   int64  
 5   gdp_per_capita_ppp  2194 non-null   float64
dtypes: float64(1), int64(1), object(4)
memory usage: 111.9+ KB


In [8]:
# Save the cleaned, long-format table as a new CSV file
df_long.to_csv("world_bank_clean_long.csv", index=False)

In [9]:
# Load the OECD raw file — no skiprows needed, headers are on row 1
df_oecd = pd.read_csv("oecd_hours_worked_raw.csv")

df_oecd.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,MEASURE,Measure,UNIT_MEASURE,Unit of measure,...,TIME_PERIOD,Time period,OBS_VALUE,Observation value,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals
0,DATAFLOW,OECD.ELS.SAE:DSD_HW@DF_AVG_ANN_HRS_WKD(1.0),Average annual hours actually worked per worker,I,AUS,Australia,HW,Hours worked,H_Y_PS,Hours per year per person,...,2025,NaN,1633.000,NaN,A,Normal value,NaN,NaN,0,Zero
1,DATAFLOW,OECD.ELS.SAE:DSD_HW@DF_AVG_ANN_HRS_WKD(1.0),Average annual hours actually worked per worker,I,AUT,Austria,HW,Hours worked,H_Y_PS,Hours per year per person,...,2017,NaN,1497.423,NaN,A,Normal value,NaN,NaN,0,Zero
2,DATAFLOW,OECD.ELS.SAE:DSD_HW@DF_AVG_ANN_HRS_WKD(1.0),Average annual hours actually worked per worker,I,AUT,Austria,HW,Hours worked,H_Y_PS,Hours per year per person,...,2018,NaN,1501.390,NaN,A,Normal value,NaN,NaN,0,Zero
3,DATAFLOW,OECD.ELS.SAE:DSD_HW@DF_AVG_ANN_HRS_WKD(1.0),Average annual hours actually worked per worker,I,AUT,Austria,HW,Hours worked,H_Y_PS,Hours per year per person,...,2019,NaN,1508.857,NaN,A,Normal value,NaN,NaN,0,Zero
4,DATAFLOW,OECD.ELS.SAE:DSD_HW@DF_AVG_ANN_HRS_WKD(1.0),Average annual hours actually worked per worker,I,AUT,Austria,HW,Hours worked,H_Y_PS,Hours per year per person,...,2020,NaN,1398.755,NaN,A,Normal value,NaN,NaN,0,Zero


In [10]:
# Keep only what we need: country name, country code, year, and the actual value
df_oecd_clean = df_oecd[["Reference area", "REF_AREA", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={
    "Reference area": "country_name",
    "REF_AREA": "country_code",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "avg_annual_hours_worked"
})

df_oecd_clean.head()

,country_name,country_code,year,avg_annual_hours_worked
0,Australia,AUS,2025,1633.000
1,Austria,AUT,2017,1497.423
2,Austria,AUT,2018,1501.390
3,Austria,AUT,2019,1508.857
4,Austria,AUT,2020,1398.755


In [11]:
df_oecd_clean["avg_annual_hours_worked"].isna().sum()

np.int64(1)

In [14]:
df_oecd_clean = df_oecd_clean[df_oecd_clean["year"].between(2017, 2025)]

In [15]:
df_oecd_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 347 entries, 0 to 346
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   country_name             347 non-null    object 
 1   country_code             347 non-null    object 
 2   year                     347 non-null    int64  
 3   avg_annual_hours_worked  346 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 11.0+ KB


In [17]:
df_oecd_clean.to_csv("oecd_clean.csv", index=False)

In [18]:
# See exactly which rows are the OECD aggregate
df_oecd_clean[df_oecd_clean["country_code"] == "OECD"]

,country_name,country_code,year,avg_annual_hours_worked
227,OECD,OECD,2017,1779.478000
228,OECD,OECD,2018,1774.946000
229,OECD,OECD,2019,1765.794000
230,OECD,OECD,2020,1686.561000
231,OECD,OECD,2021,1740.558000
232,OECD,OECD,2022,1748.950000
233,OECD,OECD,2023,1745.031000
234,OECD,OECD,2024,1741.130000
346,OECD,OECD,2025,1736.128915


In [20]:
# Drop the OECD-wide aggregate row — it's not a country, and isn't part of
# the US vs. EU comparison this project is scoped to.
df_oecd_clean = df_oecd_clean[df_oecd_clean["country_code"] != "OECD"]

df_oecd_clean.shape  # confirm row count dropped by exactly 9

(338, 4)

In [21]:
# Re-export the corrected version (OECD aggregate rows removed) as the main file
df_oecd_clean.to_csv("oecd_clean.csv", index=False)

In [22]:
# Recreate and export the version that KEEPS the OECD aggregate rows,
# by re-selecting from the original df_oecd (before any rows were dropped)
df_oecd_with_agg = df_oecd[["Reference area", "REF_AREA", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={
    "Reference area": "country_name",
    "REF_AREA": "country_code",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "avg_annual_hours_worked"
})
df_oecd_with_agg.to_csv("oecd_clean_with_agg.csv", index=False)

In [24]:
# ATTEMPT 1 (failed):
# df_hdi = pd.read_csv("un_hdi_raw.csv")
# Result: UnicodeDecodeError — 'utf-8' codec can't decode byte 0xf4 at
# position 190854 ("invalid continuation byte"). This means the file has
# at least one character (likely an accented letter in a country name)
# that isn't valid standard UTF-8 — probably introduced when Excel saved
# the file using a different text encoding on export.

# FIX: specify a more permissive encoding that accepts any byte value
df_hdi = pd.read_csv("un_hdi_raw.csv", encoding="latin-1")
df_hdi.head()

,iso3,country,hdicode,region,hdi_rank_2023,hdi_1990,hdi_1991,hdi_1992,hdi_1993,hdi_1994,...,pop_total_2014,pop_total_2015,pop_total_2016,pop_total_2017,pop_total_2018,pop_total_2019,pop_total_2020,pop_total_2021,pop_total_2022,pop_total_2023
0,AFG,Afghanistan,Low,SA,181.0,0.285,0.291,0.301,0.311,0.305,...,32.792523,33.831764,34.700612,35.688935,36.743039,37.856121,39.068979,40.000412,40.578842,41.454761
1,ALB,Albania,Very High,ECA,71.0,0.654,0.638,0.622,0.624,0.629,...,2.903749,2.898632,2.897867,2.898242,2.894231,2.885009,2.871954,2.849636,2.827608,2.811655
2,DZA,Algeria,High,AS,96.0,0.595,0.596,0.601,0.603,0.603,...,39.205030,40.019529,40.850721,41.689299,42.505035,43.294546,44.042091,44.761099,45.477390,46.164219
3,AND,Andorra,Very High,NaN,32.0,NaN,NaN,NaN,NaN,NaN,...,0.073737,0.072174,0.072181,0.073762,0.075162,0.076473,0.077380,0.078364,0.079705,0.080856
4,AGO,Angola,Medium,SSA,148.0,NaN,NaN,NaN,NaN,NaN,...,27.160770,28.157798,29.183070,30.234839,31.297155,32.375633,33.451132,34.532429,35.635029,36.749906


In [25]:
# See every column name — with 1112 columns, we need to know what's actually
# in here before deciding what to keep
print(df_hdi.columns.tolist())

['iso3', 'country', 'hdicode', 'region', 'hdi_rank_2023', 'hdi_1990', 'hdi_1991', 'hdi_1992', 'hdi_1993', 'hdi_1994', 'hdi_1995', 'hdi_1996', 'hdi_1997', 'hdi_1998', 'hdi_1999', 'hdi_2000', 'hdi_2001', 'hdi_2002', 'hdi_2003', 'hdi_2004', 'hdi_2005', 'hdi_2006', 'hdi_2007', 'hdi_2008', 'hdi_2009', 'hdi_2010', 'hdi_2011', 'hdi_2012', 'hdi_2013', 'hdi_2014', 'hdi_2015', 'hdi_2016', 'hdi_2017', 'hdi_2018', 'hdi_2019', 'hdi_2020', 'hdi_2021', 'hdi_2022', 'hdi_2023', 'le_1990', 'le_1991', 'le_1992', 'le_1993', 'le_1994', 'le_1995', 'le_1996', 'le_1997', 'le_1998', 'le_1999', 'le_2000', 'le_2001', 'le_2002', 'le_2003', 'le_2004', 'le_2005', 'le_2006', 'le_2007', 'le_2008', 'le_2009', 'le_2010', 'le_2011', 'le_2012', 'le_2013', 'le_2014', 'le_2015', 'le_2016', 'le_2017', 'le_2018', 'le_2019', 'le_2020', 'le_2021', 'le_2022', 'le_2023', 'eys_1990', 'eys_1991', 'eys_1992', 'eys_1993', 'eys_1994', 'eys_1995', 'eys_1996', 'eys_1997', 'eys_1998', 'eys_1999', 'eys_2000', 'eys_2001', 'eys_2002', 'eys

In [26]:
# This file bundles dozens of indicators (HDI, life expectancy, schooling,
# GNI, gender breakdowns, inequality-adjusted variants, CO2, population...).
# We only need the core HDI value, its tier label, and population (for
# possible bubble-sizing later) — scoped to 2017-2023, since UN data stops
# there.
hdi_year_cols = [f"hdi_{y}" for y in range(2017, 2024)]
pop_year_cols = [f"pop_total_{y}" for y in range(2017, 2024)]

keep_cols = ["iso3", "country", "hdicode"] + hdi_year_cols + pop_year_cols
df_hdi_clean = df_hdi[keep_cols]

df_hdi_clean.head()

,iso3,country,hdicode,hdi_2017,hdi_2018,hdi_2019,hdi_2020,hdi_2021,hdi_2022,hdi_2023,pop_total_2017,pop_total_2018,pop_total_2019,pop_total_2020,pop_total_2021,pop_total_2022,pop_total_2023
0,AFG,Afghanistan,Low,0.496,0.498,0.507,0.501,0.486,0.495,0.496,35.688935,36.743039,37.856121,39.068979,40.000412,40.578842,41.454761
1,ALB,Albania,Very High,0.798,0.801,0.805,0.794,0.794,0.806,0.810,2.898242,2.894231,2.885009,2.871954,2.849636,2.827608,2.811655
2,DZA,Algeria,High,0.746,0.749,0.753,0.742,0.755,0.761,0.763,41.689299,42.505035,43.294546,44.042091,44.761099,45.477390,46.164219
3,AND,Andorra,Very High,0.873,0.875,0.876,0.851,0.871,0.893,0.913,0.073762,0.075162,0.076473,0.077380,0.078364,0.079705,0.080856
4,AGO,Angola,Medium,0.610,0.611,0.611,0.610,0.609,0.615,0.616,30.234839,31.297155,32.375633,33.451132,34.532429,35.635029,36.749906


In [27]:
# Melt the HDI columns into long format
df_hdi_long = df_hdi_clean.melt(
    id_vars=["iso3", "country", "hdicode"],
    value_vars=hdi_year_cols,
    var_name="year_hdi",
    value_name="hdi_value"
)
# Clean up the year column: "hdi_2017" -> 2017
df_hdi_long["year"] = df_hdi_long["year_hdi"].str.replace("hdi_", "").astype(int)
df_hdi_long = df_hdi_long.drop(columns=["year_hdi"])

df_hdi_long.head()

,iso3,country,hdicode,hdi_value,year
0,AFG,Afghanistan,Low,0.496,2017
1,ALB,Albania,Very High,0.798,2017
2,DZA,Algeria,High,0.746,2017
3,AND,Andorra,Very High,0.873,2017
4,AGO,Angola,Medium,0.610,2017


In [28]:
# Melt the population columns into long format, same approach as HDI
df_pop_long = df_hdi_clean.melt(
    id_vars=["iso3", "country", "hdicode"],
    value_vars=pop_year_cols,
    var_name="year_pop",
    value_name="pop_total"
)
df_pop_long["year"] = df_pop_long["year_pop"].str.replace("pop_total_", "").astype(int)
df_pop_long = df_pop_long.drop(columns=["year_pop"])

df_pop_long.head()

,iso3,country,hdicode,pop_total,year
0,AFG,Afghanistan,Low,35.688935,2017
1,ALB,Albania,Very High,2.898242,2017
2,DZA,Algeria,High,41.689299,2017
3,AND,Andorra,Very High,0.073762,2017
4,AGO,Angola,Medium,30.234839,2017


In [29]:
# Merge HDI and population back into a single long table, matched on
# country code + year
df_hdi_final = df_hdi_long.merge(
    df_pop_long[["iso3", "year", "pop_total"]],
    on=["iso3", "year"],
    how="left"
)

df_hdi_final.head()

,iso3,country,hdicode,hdi_value,year,pop_total
0,AFG,Afghanistan,Low,0.496,2017,35.688935
1,ALB,Albania,Very High,0.798,2017,2.898242
2,DZA,Algeria,High,0.746,2017,41.689299
3,AND,Andorra,Very High,0.873,2017,0.073762
4,AGO,Angola,Medium,0.610,2017,30.234839


In [30]:
# ─────────────────────────────────────────────────────────────
# NOTE: There's a faster way to do the melt + merge above.
#
# Instead of melting hdi and pop_total separately (2 melts) and then
# merging them back together (1 merge) — the steps that produced
# df_hdi_long, df_pop_long, and df_hdi_final above — pandas has a
# built-in function for exactly this situation:
#
#     df_long_fast = pd.wide_to_long(
#         df_hdi_clean,
#         stubnames=["hdi", "pop_total"],
#         i=["iso3", "country", "hdicode"],
#         j="year",
#         sep="_"
#     ).reset_index()
#
# This single call replaces ALL of the following steps:
#   - the two separate .melt() calls (df_hdi_long, df_pop_long)
#   - the two .str.replace() + .astype(int) year-cleanup lines
#   - the .drop(columns=[...]) cleanup lines
#   - the final .merge() step
#
# Why I did it manually instead: this was my first time reshaping a file

In [31]:
df_hdi_final["hdi_value"].isna().sum()
df_hdi_final["pop_total"].isna().sum()

np.int64(0)

In [32]:
df_hdi_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1442 entries, 0 to 1441
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   iso3       1442 non-null   object 
 1   country    1442 non-null   object 
 2   hdicode    1365 non-null   object 
 3   hdi_value  1422 non-null   float64
 4   year       1442 non-null   int64  
 5   pop_total  1442 non-null   float64
dtypes: float64(2), int64(1), object(3)
memory usage: 67.7+ KB


In [33]:
df_hdi_final.to_csv("un_hdi_clean.csv", index=False)